In [1]:
import os, glob, duckdb, pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import average_precision_score, roc_auc_score

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.max_columns', 50)

RAW  = '../data/raw/epoch_data'
PROC = '../data/processed'

D          = '2026-09-01'
TRAIN_DATE = '2026-08-11'     # 공식 train_labels (FUT=18 고정, 그대로 사용)
HOLD_DATE  = '2026-08-25'     # 공식 train_labels (FUT=18 고정, 그대로 사용)
FUT_FINAL  = 26               # 본선 채점 기준. 우리가 직접 만드는 추가 기준일에 적용

con = duckdb.connect()
con.sql("SET memory_limit = '4GB'")
for p in sorted(glob.glob(f'{RAW}/*.csv')):
    name = os.path.splitext(os.path.basename(p))[0]
    con.sql(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_csv('{p}', header=true, auto_detect=true)")
con.sql(f"CREATE OR REPLACE VIEW video_snapshots AS SELECT * FROM '{PROC}/video_snapshots.parquet'")
print('준비 완료')

준비 완료


In [2]:
def make_features(Dref):
    """기준일 Dref 시점 피처. CUT_HOURS 제한 없음 (09/01 이후 관측값도 past 영상이면 사용 — 공지로 확인됨)"""
    return con.sql(f"""
    WITH v5 AS (
        SELECT channel_id,
               COUNT(*) AS n_long_5d,
               MEDIAN(view_5d) AS past_med,
               MEDIAN(view_5d) FILTER (WHERE published_at >= TIMESTAMP '{Dref}' - INTERVAL 14 DAY) AS recent14_med,
               MEDIAN(like_5d / NULLIF(view_5d, 0)) AS like_rate,
               STDDEV(LN(1 + view_5d)) AS log_view_std
        FROM video_5d_views
        WHERE is_shorts = 0 AND view_5d IS NOT NULL
          AND published_at < TIMESTAMP '{Dref}'          -- CUT_HOURS 제거, published_at만 기준
        GROUP BY 1
    ),
    up AS (
        SELECT channel_id,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 14 DAY) AS n_long_14,
               COUNT(*) FILTER (WHERE is_shorts = 1 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 14 DAY) AS n_short_14,
               DATE_DIFF('hour', MAX(published_at) FILTER (WHERE is_shorts = 0), TIMESTAMP '{Dref}') / 24.0 AS days_since_long
        FROM videos WHERE published_at < TIMESTAMP '{Dref}' GROUP BY 1
    ),
    games AS (
        SELECT v.channel_id,
               COUNT(DISTINCT vg.app_id) AS n_games
        FROM videos v LEFT JOIN video_games vg USING (video_id)
        WHERE v.published_at < TIMESTAMP '{Dref}'
        GROUP BY 1
    )
    SELECT * FROM v5 LEFT JOIN up USING (channel_id) LEFT JOIN games USING (channel_id)
    """).df()

def make_labels(Dref, FUT=FUT_FINAL):
    """라벨 재현 (이제 FUT=26 기본값). CUT_HOURS 없이 published_at만으로 past/future 구분 (공지로 확인됨)"""
    g = con.sql(f"""
        WITH agg AS (
          SELECT channel_id,
                 MEDIAN(view_5d) FILTER (WHERE published_at < TIMESTAMP '{Dref}') AS past_score,
                 COUNT(view_5d)  FILTER (WHERE published_at < TIMESTAMP '{Dref}') AS n_past,
                 COALESCE(MEDIAN(view_5d) FILTER (
                     WHERE published_at >= TIMESTAMP '{Dref}'
                       AND published_at <  TIMESTAMP '{Dref}' + INTERVAL {FUT} DAY), 0) AS future_score
          FROM video_5d_views
          WHERE is_shorts = 0 AND view_5d IS NOT NULL
          GROUP BY 1
        ),
        scored AS (
          SELECT channel_id, LN(1+future_score)-LN(1+past_score) AS growth
          FROM agg WHERE n_past >= 3 AND past_score >= 100
        )
        SELECT channel_id,
               CASE WHEN growth >= QUANTILE_CONT(growth, 0.8) OVER () THEN 1 ELSE 0 END AS target
        FROM scored
    """).df()
    g['ref_date'] = Dref
    return g

In [3]:
EXTRA_DATES = ['2026-07-18', '2026-07-21', '2026-07-24', '2026-07-27', '2026-07-30', '2026-08-02', '2026-08-05']
extra_labels = pd.concat([make_labels(d) for d in EXTRA_DATES], ignore_index=True)
print(extra_labels.groupby('ref_date')['target'].agg(['size', 'mean']))

            size      mean
ref_date                  
2026-07-18   139  0.201439
2026-07-21   159  0.201258
2026-07-24   234  0.200855
2026-07-27   389  0.200514
2026-07-30   448  0.200893
2026-08-02   509  0.200393
2026-08-05   537  0.201117


In [4]:
# ===== 우리가 만든 FUT=26 라벨에 피처 붙이기 =====
extra_pieces = [extra_labels[extra_labels.ref_date == d].merge(make_features(d), on='channel_id', how='left')
                for d in EXTRA_DATES]
extra_f = pd.concat(extra_pieces, ignore_index=True)

# ===== 공식 train_labels (08-11, 08-25, FUT=18 고정) 불러와서 피처 붙이기 =====
tl = pd.read_csv(f'{RAW}/train_labels.csv')
tl['channel_id'] = tl['row_id'].str.rsplit('_', n=1).str[0]
tl['ref_date']   = tl['row_id'].str.rsplit('_', n=1).str[1]
tl = tl.rename(columns={'target': 'target'})   # 컬럼명 통일 확인용

official_f = pd.concat([
    tl[tl.ref_date == d].merge(make_features(d), on='channel_id', how='left')
    for d in [TRAIN_DATE, HOLD_DATE]
], ignore_index=True)

# ===== 전체 합치기 =====
all_data = pd.concat([official_f, extra_f], ignore_index=True)
print('전체 행 수:', len(all_data))
print(all_data.groupby('ref_date')['target'].agg(['size', 'mean']))
print()
print('결측 확인:')
print(all_data[['past_med','recent14_med','n_games']].isna().mean().round(3))

전체 행 수: 3682
            size      mean
ref_date                  
2026-07-18   139  0.201439
2026-07-21   159  0.201258
2026-07-24   234  0.200855
2026-07-27   389  0.200514
2026-07-30   448  0.200893
2026-08-02   509  0.200393
2026-08-05   537  0.201117
2026-08-11   601  0.201331
2026-08-25   666  0.201201

결측 확인:
past_med        0.000
recent14_med    0.014
n_games         0.000
dtype: float64


In [5]:
# ===== 모델링: holdout=8/5(FUT=26), train_all=나머지 =====
HOLD = '2026-08-05'
def prep(df):
    X = pd.DataFrame(index=df.index)
    for c in ['past_med','recent14_med','n_long_5d','n_long_14','n_short_14','n_games']:
        X[c] = np.log1p(df[c].astype(float).clip(lower=0))
    X['like_rate'] = df['like_rate'].astype(float)
    X['log_view_std'] = df['log_view_std'].astype(float)
    X['days_since_long'] = df['days_since_long'].astype(float)
    X['has_game'] = (df['n_games'].fillna(0) > 0).astype(int)
    X['recent_none'] = ((df['n_long_14'].fillna(0) == 0)).astype(int)
    X['recent_unmeasured'] = ((df['n_long_14'].fillna(0) > 0) & df['recent14_med'].isna()).astype(int)
    X['momentum14'] = np.log1p(df['recent14_med'].astype(float)) - np.log1p(df['past_med'].astype(float))
    X.loc[df['n_long_14'].fillna(0) < 2, 'momentum14'] = np.nan
    return X

train_all = all_data[all_data.ref_date != HOLD].reset_index(drop=True)
hold      = all_data[all_data.ref_date == HOLD].reset_index(drop=True)
Xtr, ytr, Xho, yho = prep(train_all), train_all['target'], prep(hold), hold['target']

def make_pipe(C=1.0):
    return Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()),
                     ('lr', LogisticRegression(C=C, max_iter=2000))])

# 날짜 단위 CV (같은 기준일 통째로 검증 — 채널 중복 누수 방지)
dates = train_all['ref_date'].values
cv_scores = []
for d in sorted(set(dates)):
    m = dates == d
    p = make_pipe().fit(Xtr[~m], ytr[~m]).predict_proba(Xtr[m])[:, 1]
    cv_scores.append(average_precision_score(ytr[m], p))
    print(d, round(cv_scores[-1], 4), 'base', round(ytr[m].mean(), 3))
print('날짜별 CV PR-AUC 평균 %.4f ± %.4f' % (np.mean(cv_scores), np.std(cv_scores)))

pipe = make_pipe().fit(Xtr, ytr)
ph = pipe.predict_proba(Xho)[:, 1]
print('holdout(8/5) PR-AUC %.4f / ROC-AUC %.4f / base %.3f' % (average_precision_score(yho, ph), roc_auc_score(yho, ph), yho.mean()))


2026-07-18 0.3714 base 0.201
2026-07-21 0.4386 base 0.201
2026-07-24 0.3555 base 0.201
2026-07-27 0.3578 base 0.201
2026-07-30 0.3487 base 0.201
2026-08-02 0.397 base 0.2
2026-08-11 0.3468 base 0.201
2026-08-25 0.3663 base 0.201
날짜별 CV PR-AUC 평균 0.3728 ± 0.0290
holdout(8/5) PR-AUC 0.3651 / ROC-AUC 0.6794 / base 0.201


In [6]:
# ===== 업로드 빈도 피처 추가 & 비교 =====
def freq_features(Dref):
    return con.sql(f"""
    WITH l AS (SELECT channel_id, published_at FROM videos WHERE is_shorts = 0 AND published_at < TIMESTAMP '{Dref}'),
    gaps AS (
        SELECT channel_id, DATE_DIFF('hour', LAG(published_at) OVER (PARTITION BY channel_id ORDER BY published_at), published_at)/24.0 AS gap
        FROM l)
    SELECT l.channel_id,
           COUNT(*) FILTER (WHERE published_at >= TIMESTAMP '{Dref}' - INTERVAL 30 DAY) AS n_long_30,
           COUNT(*) FILTER (WHERE published_at >= TIMESTAMP '{Dref}' - INTERVAL 60 DAY) AS n_long_60,
           COUNT(*) FILTER (WHERE published_at >= TIMESTAMP '{Dref}' - INTERVAL 90 DAY) AS n_long_90,
           DATE_DIFF('hour', MIN(published_at), TIMESTAMP '{Dref}')/24.0 AS span_days,
           (SELECT AVG(gap) FROM gaps g WHERE g.channel_id = l.channel_id) AS gap_mean,
           (SELECT STDDEV(gap) FROM gaps g WHERE g.channel_id = l.channel_id) AS gap_std
    FROM l GROUP BY l.channel_id
    """).df()

fr = pd.concat([freq_features(d).assign(ref_date=d) for d in sorted(all_data.ref_date.unique())], ignore_index=True)
all2 = all_data.merge(fr, on=['channel_id', 'ref_date'], how='left')

def prep2(df):
    X = prep(df)
    for c in ['n_long_30', 'n_long_60', 'n_long_90']:
        X[c] = np.log1p(df[c].fillna(0))
    X['rate_30'] = df['n_long_30'].fillna(0) / 30
    X['rate_90'] = df['n_long_90'].fillna(0) / 90
    X['rate_all'] = df['n_long_5d'] / df['span_days'].clip(lower=1)
    X['gap_mean'] = np.log1p(df['gap_mean'])
    X['gap_cv'] = df['gap_std'] / df['gap_mean'].clip(lower=0.5)
    X['overdue'] = df['days_since_long'] / df['gap_mean'].clip(lower=0.5)   # 평소 간격 대비 쉰 정도
    return X

def evaluate(df, prepf, C=1.0, name=''):
    tr, ho = df[df.ref_date != HOLD].reset_index(drop=True), df[df.ref_date == HOLD].reset_index(drop=True)
    Xt, yt = prepf(tr), tr['target']; Xh, yh = prepf(ho), ho['target']
    ds, sc = tr['ref_date'].values, []
    for d in sorted(set(ds)):
        m = ds == d
        sc.append(average_precision_score(yt[m], make_pipe(C).fit(Xt[~m], yt[~m]).predict_proba(Xt[m])[:, 1]))
    ph = make_pipe(C).fit(Xt, yt).predict_proba(Xh)[:, 1]
    print(f'{name:12s} CV {np.mean(sc):.4f} ± {np.std(sc):.3f} | holdout PR {average_precision_score(yh, ph):.4f} ROC {roc_auc_score(yh, ph):.4f}')

evaluate(all2, prep,  name='기존')
evaluate(all2, prep2, name='+업로드빈도')
evaluate(all2, prep2, C=0.1, name='+빈도 C=0.1')


기존           CV 0.3728 ± 0.029 | holdout PR 0.3651 ROC 0.6794
+업로드빈도       CV 0.3724 ± 0.023 | holdout PR 0.3658 ROC 0.6882


+빈도 C=0.1    CV 0.3717 ± 0.023 | holdout PR 0.3671 ROC 0.6911


In [7]:
# ===== LR vs LightGBM vs 앙상블 (날짜 단위 CV + holdout 8/5) =====
import lightgbm as lgb
from scipy.stats import rankdata

def lgb_model(seed=0, **kw):
    p = dict(n_estimators=200, learning_rate=0.03, num_leaves=4, max_depth=3, min_child_samples=60,
             subsample=0.7, subsample_freq=1, colsample_bytree=0.7, reg_lambda=10, random_state=seed, verbose=-1)
    p.update(kw)
    return lgb.LGBMClassifier(**p)

def fit_predict(Xa, ya, Xb):
    p_lr = make_pipe(0.1).fit(Xa, ya).predict_proba(Xb)[:, 1]
    p_gb = np.mean([lgb_model(s).fit(Xa, ya).predict_proba(Xb)[:, 1] for s in range(5)], axis=0)
    r = lambda p: rankdata(p) / len(p)
    return {'LR': p_lr, 'LGB': p_gb, 'ENS(rank)': (r(p_lr) + r(p_gb)) / 2}

tr = all2[all2.ref_date != HOLD].reset_index(drop=True)
ho = all2[all2.ref_date == HOLD].reset_index(drop=True)
Xt, yt, Xh, yh = prep2(tr), tr['target'], prep2(ho), ho['target']
ds, res = tr['ref_date'].values, {}
for d in sorted(set(ds)):
    m = ds == d
    for k, p in fit_predict(Xt[~m], yt[~m], Xt[m]).items():
        res.setdefault(k, []).append(average_precision_score(yt[m], p))
hp = fit_predict(Xt, yt, Xh)
for k in res:
    print(f'{k:10s} CV {np.mean(res[k]):.4f} ± {np.std(res[k]):.3f} | holdout PR {average_precision_score(yh, hp[k]):.4f}')


LR         CV 0.3717 ± 0.023 | holdout PR 0.3671
LGB        CV 0.3952 ± 0.052 | holdout PR 0.4441
ENS(rank)  CV 0.3915 ± 0.036 | holdout PR 0.4064
